# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
# A runner object is needed to execute the agent. It handles the conversation history and other details.
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?") # async python function

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient fruit with benefits:
    
    - Nutrients: good source of potassium, vitamin B6, vitamin C, and fiber.
    - Health benefits: supports heart health and digestion; provides quick, lasting energy.
    - Fiber: helps with satiety and gut health; unripe bananas have more resistant starch.
    - Sugar: moderate natural sugars; about 14 g per medium banana.
    - Weight/diabetes considerations: okay in moderation; pair with protein/fat if watching sugar impact.
    
    Bottom line: a nutritious, fiber-rich snack when eaten as part of a balanced diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
# Make it chatgpt style and stream the response
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance( # Only print the delta events, not the full response events
        event.data, ResponseTextDeltaEvent
    ):
        # Print streaming response without a newline and flush the output buffer to show it immediately
        print(event.data.delta, end="", flush=True)

Bananas are healthy for most people. Key points:

- Nutrients: good in potassium, vitamin C, vitamin B6, dietary fiber, and some resistant starch (especially when unripe).
- Benefits: supports heart health, blood pressure, digestion, and provides quick energy (natural sugars + fiber).
- Glycemic impact: moderate GI; portion size matters if you’re watching blood sugar.
- Cautions: they’re relatively high in carbs/sugar, so consider portions if on a low-carb diet. Rare allergies.
- Ripeness: unripe = higher resistant starch (fiber); ripe = sweeter, more sugar but still nutritious.
- Tips: store at room temp to ripen; refrigerate to extend shelf life (peels darken but flesh stays good).

Bottom line: a convenient, nutrient-dense fruit that fits most healthy diets in moderate portions.

_Good Job!_